<div style="background:linear-gradient(120deg,#0b3a67 0%,#0f6cbd 60%,#2aa0a4 100%);background-color:#0f6cbd;border-radius:12px;padding:22px 28px;color:#ffffff"><table style="border:none;border-collapse:collapse;width:100%;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0;vertical-align:middle;text-align:left;color:#ffffff"><div style="font-size:13px;letter-spacing:2px;text-transform:uppercase;opacity:0.85">Microsoft Fabric &middot; Lakehouse &middot; Component</div><div style="font-size:34px;font-weight:700;line-height:1.2;margin:2px 0 6px 0">Backup Form</div><div style="font-size:15px;line-height:1.5;max-width:520px">The input screen for a backup: name the source and staging lakehouses, browse the write-once container to pick or name the backup folder, and choose the options. It checks the inputs and looks up the IDs.</div></td><td style="border:none;padding:0 0 0 20px;vertical-align:middle;text-align:right"></td></tr></table></div><div style="margin:12px 2px 0 2px;font-size:14px;line-height:1.6"><b>This notebook is not opened to do a job.</b> It is loaded by the Backup Utility notebook, which shows the form and then uses what was entered.<br/><span style="display:inline-block;background:#107c10;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Reads only</span> &nbsp; <span style="display:inline-block;background:#107c10;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Copies and deletes nothing</span> &nbsp; <span style="display:inline-block;background:#5c2e91;color:#ffffff;border-radius:11px;padding:1px 10px;font-size:12px;font-weight:600">Edit only to change the form</span></div>


<div style="border-left:6px solid #5c2e91;border-radius:6px;padding:10px 16px;margin:14px 0 4px 0;background:rgba(127,127,127,0.09)"><table style="border:none;border-collapse:collapse;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0 14px 0 0;vertical-align:top"></td><td style="border:none;padding:0;vertical-align:top;text-align:left"><div style="font-size:12px;letter-spacing:1.5px;text-transform:uppercase;color:#5c2e91;font-weight:700">Part 1</div><div style="font-size:20px;font-weight:600;margin:1px 0 6px 0">Form code</div><div style="font-size:14px;line-height:1.55">Defines everything the form needs. Nothing runs until the form is shown in Part 2.<ul style="margin:6px 0 6px 0"><li><b>Lookups</b> (<code>fabric_list</code>, <code>find_by_name</code>): find a workspace or lakehouse by name, as the signed-in person, and return its ID.</li><li><b>Container browsing</b> (<code>browse_folders</code>): list the folders in the container and mark the ones with no backups yet.</li><li><b>Checking</b> (<code>resolve_backup_inputs</code>): finds the four IDs, refuses a staging lakehouse that is the source, and tests that the container can be read.</li><li><b>The form</b> (<code>BackupInputForm</code>): the boxes, the folder list, two tick boxes and two buttons, <b>Browse</b> and <b>Get IDs</b>.</li></ul><b>Expect:</b> no output. <b>Result for later cells:</b> after <b>Get IDs</b> succeeds, the values are in <code>form.params</code>; any change on the form clears them until the button is pressed again. A new folder is not created here: the backup creates it when it writes.</div></td></tr></table></div>


In [ ]:
import html, re, time
import requests
import ipywidgets as W
import notebookutils

FABRIC_API = "https://api.fabric.microsoft.com/v1/"
NEW_FOLDER = "__new__"


def fabric_list(path):
    """All pages of a Fabric REST list call, as the signed-in user."""
    headers = {"Authorization": "Bearer " + notebookutils.credentials.getToken("pbi")}
    items, token = [], None
    while True:
        for attempt in range(1, 6):
            response = requests.get(FABRIC_API + path, headers=headers, timeout=60,
                                    params={"continuationToken": token} if token else None)
            if response.status_code in (429, 500, 502, 503, 504) and attempt < 5:
                time.sleep(int(response.headers.get("Retry-After", 2 ** attempt)))
                continue
            break
        response.raise_for_status()
        page = response.json()
        items += page.get("value", [])
        token = page.get("continuationToken")
        if not token:
            return items


def find_by_name(items, name, kind, where=""):
    """The one item whose displayName equals `name` (upper/lower case ignored)."""
    name = name.strip()
    if not name:
        raise ValueError(f"{kind}: nothing was typed.")
    hits = [i for i in items if i["displayName"].lower() == name.lower()]
    if len(hits) == 1:
        return hits[0]
    if len(hits) > 1:
        raise ValueError(f'{kind}: {len(hits)} items are named "{name}"{where}. Names must be unique.')
    close = sorted(i["displayName"] for i in items if name.lower() in i["displayName"].lower())[:5]
    hint = f" Did you mean: {', '.join(close)}?" if close else ""
    raise ValueError(f'{kind}: "{name}" was not found{where}, or you have no access to it.{hint}')


def container_address(text):
    """Accepts abfss://container@account.dfs.core.windows.net[/folder]
    or https://account.dfs.core.windows.net/container[/folder]. Returns the abfss form."""
    text = text.strip().rstrip("/")
    if not text:
        raise ValueError("Backup container: nothing was typed.")
    m = re.fullmatch(r"https://([^./]+)\.(?:dfs|blob)\.core\.windows\.net/([^/]+)(/.*)?", text)
    if m:
        text = f"abfss://{m.group(2)}@{m.group(1)}.dfs.core.windows.net{m.group(3) or ''}"
    if not re.fullmatch(r"abfss://[^@/]+@[^./]+\.dfs\.core\.windows\.net(/.*)?", text):
        raise ValueError("Backup container must look like "
                         "abfss://<container>@<account>.dfs.core.windows.net")
    return text


def browse_folders(container):
    """Folders directly under the typed address. Returns (address, [(label, folder)]).
    The folder '' means the address itself already holds backups."""
    address = container_address(container)
    try:
        items = notebookutils.fs.ls(address)
    except Exception as exc:
        raise ValueError(f"Could not read {address}. Check the address and your access. "
                         + " ".join(str(exc).split())[:200])
    names = sorted(i.name.rstrip("/") for i in items if i.isDir)
    options = [("(this location: it already holds backups)", "")] if "backups" in names else []
    for name in names:
        if name == "backups":
            continue
        has_backups = notebookutils.fs.exists(f"{address}/{name}/backups")
        options.append((name + ("" if has_backups else "   (no backups yet)"), name))
    return address, options


def resolve_backup_inputs(source_workspace, source_lakehouse, staging_workspace, staging_lakehouse, container):
    """Turns the five typed values into IDs. Reads only; changes nothing."""
    workspaces = fabric_list("workspaces")
    src_ws = find_by_name(workspaces, source_workspace, "Source workspace")
    src_lh = find_by_name(fabric_list(f"workspaces/{src_ws['id']}/lakehouses"), source_lakehouse,
                          "Source lakehouse", f' in workspace "{src_ws["displayName"]}"')
    stg_ws = find_by_name(workspaces, staging_workspace, "Staging workspace")
    stg_lh = find_by_name(fabric_list(f"workspaces/{stg_ws['id']}/lakehouses"), staging_lakehouse,
                          "Staging lakehouse", f' in workspace "{stg_ws["displayName"]}"')
    if (src_ws["id"], src_lh["id"]) == (stg_ws["id"], stg_lh["id"]):
        raise ValueError("The staging lakehouse must not be the source lakehouse.")
    address = container_address(container)
    try:
        notebookutils.fs.ls(re.match(r"abfss://[^/]+", address).group(0))     # the container itself
        reachable = "yes"
    except Exception as exc:
        reachable = "NO: " + " ".join(str(exc).split())[:200]
    return {
        "source_workspace_name": src_ws["displayName"], "source_workspace_id": src_ws["id"],
        "source_lakehouse_name": src_lh["displayName"], "source_lakehouse_id": src_lh["id"],
        "staging_workspace_name": stg_ws["displayName"], "staging_workspace_id": stg_ws["id"],
        "staging_lakehouse_name": stg_lh["displayName"], "staging_lakehouse_id": stg_lh["id"],
        "backup_container": address, "backup_container_reachable": reachable,
    }


class BackupInputForm:
    """Text boxes for the names, Browse to pick or name the backup folder, two tick boxes, one button.
    After a successful press of Get IDs, the IDs and choices are in `.params`. The form changes nothing."""

    def __init__(self):
        wide, style = W.Layout(width="720px"), {"description_width": "190px"}
        def box(label, hint):
            return W.Text(description=label, placeholder=hint, layout=wide, style=style)
        self.source_workspace = box("Source workspace name", "workspace that holds the lakehouse to back up")
        self.source_lakehouse = box("Source lakehouse name", "lakehouse to back up")
        self.staging_workspace = box("Staging workspace name", "workspace that holds the staging lakehouse")
        self.staging_lakehouse = box("Staging lakehouse name", "temporary working lakehouse, not the source")
        self.container = W.Text(description="Backup container", style=style, layout=W.Layout(width="584px"),
                                placeholder="abfss://<container>@<account>.dfs.core.windows.net")
        self.browse_button = W.Button(description="Browse", layout=W.Layout(width="130px"))
        self.folder = W.Dropdown(description="Backup folder", options=[], layout=wide, style=style)
        self.new_folder = box("New folder name", "only for a new folder; blank = <source workspace>-<source lakehouse>")
        self.copy_files = W.Checkbox(value=True, description="Include everything under Files",
                                     indent=False, layout=wide)
        self.verify_counts = W.Checkbox(value=True, description="Also compare row counts (slower)",
                                        indent=False, layout=wide)
        self.button = W.Button(description="Get IDs", button_style="info", layout=W.Layout(width="200px"))
        self.result = W.HTML()
        self.params = None
        self._address = None                    # the address the folder list was read from
        self._loading = False
        self.button.on_click(lambda _b: self.submit())
        self.browse_button.on_click(lambda _b: self.browse())
        self.container.observe(self._container_edited, names="value")
        # any change after Get IDs clears the result, so it cannot go stale
        for w in (self.source_workspace, self.source_lakehouse, self.staging_workspace, self.staging_lakehouse,
                  self.folder, self.new_folder, self.copy_files, self.verify_counts):
            w.observe(self._changed, names="value")
        self.box = W.VBox([self.source_workspace, self.source_lakehouse, self.staging_workspace,
                           self.staging_lakehouse, W.HBox([self.container, self.browse_button]),
                           self.folder, self.new_folder, self.copy_files, self.verify_counts,
                           self.button, self.result])

    def show(self):
        display(self.box)

    def _say(self, colour, body):
        self.result.value = (f'<div style="border-left:5px solid {colour};padding:8px 12px;'
                             f'margin:8px 0;font-size:14px">{body}</div>')

    def _changed(self, _change):
        if self._loading:
            return
        if self.params is not None:
            self.params = None
            self._say("#0078d4", "A value changed. Press <b>Get IDs</b> again.")

    def _set_folders(self, options):
        self._loading = True
        try:
            self.folder.options = options
            self.folder.value = options[0][1] if options else None
        finally:
            self._loading = False

    def _container_edited(self, _change):
        """The folder list belongs to the address that was browsed; clear it when the address changes."""
        self._address = None
        self._set_folders([])
        self._changed(None)

    def browse(self):
        """What Browse does: lists the folders in the container. Reads only.
        Can also be called from a cell: form.browse()"""
        self.params = None
        self.browse_button.disabled = True
        self._say("#0078d4", "Reading the container ...")
        try:
            address, options = browse_folders(self.container.value)
            self._address = address
            self._set_folders(options + [("(new folder: type its name below)", NEW_FOLDER)])
            self._say("#0078d4", f"Found {len(options)} folder(s). Choose one, or choose the new-folder line "
                                 "and type a name. Then press <b>Get IDs</b>.")
        except Exception as exc:
            self._address = None
            self._set_folders([])
            self._say("#d13438", "<b>Could not browse.</b> " + html.escape(str(exc)))
            print("COULD NOT BROWSE:", exc)
        finally:
            self.browse_button.disabled = False

    def _target(self, source_workspace_name, source_lakehouse_name):
        """(full backup address, folder, is_new). Without Browse, the typed address is used as it is."""
        if self._address is None:
            return container_address(self.container.value), "", False
        choice = self.folder.value
        if choice != NEW_FOLDER:
            return self._address + ("/" + choice if choice else ""), choice, False
        name = self.new_folder.value.strip().strip("/")
        if not name:
            name = re.sub(r"[^A-Za-z0-9._-]+", "-", f"{source_workspace_name}-{source_lakehouse_name}").strip("-")
        if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", name):
            raise ValueError("New folder name may only contain letters, digits, dot, dash and underscore.")
        existing = [v for _l, v in self.folder.options]
        if name in existing:
            return f"{self._address}/{name}", name, False        # it already exists: use it
        return f"{self._address}/{name}", name, True

    def submit(self):
        """What the button does. Can also be called from a cell: form.submit()"""
        self.params = None
        self.button.disabled = True
        self._say("#0078d4", "Looking up the names ...")
        try:
            p = resolve_backup_inputs(self.source_workspace.value, self.source_lakehouse.value,
                                      self.staging_workspace.value, self.staging_lakehouse.value,
                                      self.container.value)
            address, folder, is_new = self._target(p["source_workspace_name"], p["source_lakehouse_name"])
        except Exception as exc:
            self._say("#d13438", "<b>Not resolved.</b> " + html.escape(str(exc)))
            print("NOT RESOLVED:", exc)
            return None
        finally:
            self.button.disabled = False
        p["backup_container"] = address          # full address, folder included: pass this to the backup
        p["backup_folder"] = folder
        p["backup_folder_is_new"] = is_new
        p["copy_files"] = bool(self.copy_files.value)
        p["verify_counts"] = bool(self.verify_counts.value)
        self.params = p
        rows = [("Source workspace", p["source_workspace_name"], p["source_workspace_id"]),
                ("Source lakehouse", p["source_lakehouse_name"], p["source_lakehouse_id"]),
                ("Staging workspace", p["staging_workspace_name"], p["staging_workspace_id"]),
                ("Staging lakehouse", p["staging_lakehouse_name"], p["staging_lakehouse_id"]),
                ("Backup will be written to", p["backup_container"] + "/backups/<timestamp>",
                 "reachable: " + p["backup_container_reachable"]),
                ("Backup folder", folder or "(as typed)",
                 "NEW: the backup creates it" if is_new else "existing"),
                ("Include Files", "yes" if p["copy_files"] else "no", ""),
                ("Compare row counts", "yes" if p["verify_counts"] else "no", "")]
        cells = "".join("<tr>" + "".join(f'<td style="padding:3px 14px 3px 0">{html.escape(str(c))}</td>'
                                         for c in r) + "</tr>" for r in rows)
        ok = p["backup_container_reachable"] == "yes"
        self._say("#107c10" if ok else "#c19c00",
                  ("<b>All names found.</b>" if ok else
                   "<b>Names found, but the container could not be read.</b> Check the address and your access.")
                  + f'<table style="margin-top:6px;font-size:13px">{cells}</table>')
        for key, value in p.items():
            print(f"{key:28} = {value}")
        return p


<div style="border-left:6px solid #0f6cbd;border-radius:6px;padding:10px 16px;margin:14px 0 4px 0;background:rgba(127,127,127,0.09)"><table style="border:none;border-collapse:collapse;margin:0"><tr style="border:none;background:none"><td style="border:none;padding:0 14px 0 0;vertical-align:top"></td><td style="border:none;padding:0;vertical-align:top;text-align:left"><div style="font-size:12px;letter-spacing:1.5px;text-transform:uppercase;color:#0f6cbd;font-weight:700">Part 2</div><div style="font-size:20px;font-weight:600;margin:1px 0 6px 0">Show the form</div><div style="font-size:14px;line-height:1.55">Creates the form and displays it. This is the cell whose output the person works in.<br/><b>To use it:</b> type the four names and the container address, press <b>Browse</b> to pick a folder or name a new one, set the tick boxes, then press <b>Get IDs</b>.<br/><b>Expect:</b> a green line <i>All names found</i> with each name, its ID and where the backup will be written, or a red line naming what was not found.<br/><b>If a button does nothing:</b> run <code>form.browse()</code> or <code>form.submit()</code> in a cell; they do the same as the buttons.</div></td></tr></table></div>


In [ ]:
form = BackupInputForm()
form.show()